In [1]:
#!pip install pycocotools
#!pip install transformers timm

In [7]:
import sys
sys.path.append("C:/Users/User/Downloads/GroundingDINO-main/GroundingDINO-main")

In [9]:
from groundingdino.util.inference import load_model, load_image,predict,annotate
import os
import cv2
import urllib.request

In [13]:
weights_dir="C:/ai_project01/GroundingDINO-main/groundingdino/weights"
os.makedirs(weights_dir,exist_ok=True)
url ="https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth"
file_path=os.path.join(weights_dir,"groundingdino_swint_ogc.pth")
print("다운중")
urllib.request.urlretrieve(url,file_path)
print(f"저장완료{file_path}")

다운중
저장완료C:/ai_project01/GroundingDINO-main/groundingdino/weights\groundingdino_swint_ogc.pth


In [18]:
# config_path는 모델의 구조(레이어 수, 백본 등)를 정의한 설정 파일의 경로입니다.
config_path = "C:/ai_project01/GroundingDINO-main/groundingdino/config/GroundingDINO_SwinT_OGC.py"

# weight_path는 사전 학습된 파라미터(.pth 파일)의 경로입니다.
weight_path = "C:/ai_project01/GroundingDINO-main/groundingdino/weights/groundingdino_swint_ogc.pth"



model = load_model(config_path, weight_path)


final text_encoder_type: bert-base-uncased


In [51]:
from pathlib import Path

image_folder = "C:/ai_project01/generate_images"
image_list = []

# barrier_scene_4k_01 ~ 08
for i in range(1, 9):
    image_list.append(f"barrier_scene_4k_{i:02d}.png")

# barrier_scene_4k_food_01 ~ 08
for i in range(1, 9):
    image_list.append(f"barrier_scene_4k_food_{i:02d}.png")

# barrier_scene_4k_pizza_01 ~ 20
for i in range(1, 21):
    image_list.append(f"barrier_scene_4k_pizza_{i:02d}.png")

# barrier_scene_4k_road_01 ~ 50
for i in range(1, 51):
    image_list.append(f"barrier_scene_4k_road_{i:02d}.png")

# 파일 경로 리스트 생성 (슬래시 `/`로 표기)
image_paths = [str(Path(image_folder, fname).as_posix()) for fname in image_list]

# 예시 출력 (처음 10개만)
for path in image_paths[:10]:
    print(path)

C:/ai_project01/generate_images/barrier_scene_4k_01.png
C:/ai_project01/generate_images/barrier_scene_4k_02.png
C:/ai_project01/generate_images/barrier_scene_4k_03.png
C:/ai_project01/generate_images/barrier_scene_4k_04.png
C:/ai_project01/generate_images/barrier_scene_4k_05.png
C:/ai_project01/generate_images/barrier_scene_4k_06.png
C:/ai_project01/generate_images/barrier_scene_4k_07.png
C:/ai_project01/generate_images/barrier_scene_4k_08.png
C:/ai_project01/generate_images/barrier_scene_4k_food_01.png
C:/ai_project01/generate_images/barrier_scene_4k_food_02.png


In [66]:
text_prompt = "person  . car .  cooter . traffic light . bulding"
box_threshold = 0.3
text_threshold = 0.25

In [73]:
for image_path in image_paths:
    # 파일 존재 확인 (생략해도 되지만 권장)
    if not os.path.isfile(image_path):
        print(f"파일 없음: {image_path}")
        continue

    # 이미지 로드 (Grounding DINO에 특화된 함수 사용)
    image_source, image_tensor = load_image(image_path)
    
    # 예측 수행
    boxes, logits, phrases = predict(
        model=model,
        image=image_tensor,  # 경로가 아니라 tensor여야 함
        caption=text_prompt,
        box_threshold=box_threshold,
        text_threshold=text_threshold
    )

    # (옵션) 결과 출력 또는 저장
    annotated = cv2.cvtColor(image_source.copy(), cv2.COLOR_RGB2BGR)
    image_height, image_width = image_source.shape[:2]

    for box, label in zip(boxes, phrases):
        cx, cy, w, h = box
        cx *= image_width
        cy *= image_height
        w *= image_width
        h *= image_height

        x_min = int(cx - w / 2)
        y_min = int(cy - h / 2)
        x_max = int(cx + w / 2)
        y_max = int(cy + h / 2)

        cv2.rectangle(annotated, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
        cv2.putText(
            annotated, label, (x_min, y_min - 10),
            cv2.FONT_HERSHEY_SIMPLEX, fontScale=0.7,
            color=(0, 255, 0), thickness=2
        )

    # 결과 이미지 저장: 원본 파일명에 맞춰서 따로 저장
    file_name = os.path.basename(image_path)
    output_image_path = os.path.join(output_folder, f"annotated_{file_name}")
    cv2.imwrite(output_image_path, annotated)
    print(f"저장 완료: {output_image_path}")

저장 완료: C:/ai_project01/GroundingDINO_labels\annotated_barrier_scene_4k_01.png
저장 완료: C:/ai_project01/GroundingDINO_labels\annotated_barrier_scene_4k_02.png
저장 완료: C:/ai_project01/GroundingDINO_labels\annotated_barrier_scene_4k_03.png
저장 완료: C:/ai_project01/GroundingDINO_labels\annotated_barrier_scene_4k_04.png
저장 완료: C:/ai_project01/GroundingDINO_labels\annotated_barrier_scene_4k_05.png
저장 완료: C:/ai_project01/GroundingDINO_labels\annotated_barrier_scene_4k_06.png
저장 완료: C:/ai_project01/GroundingDINO_labels\annotated_barrier_scene_4k_07.png
저장 완료: C:/ai_project01/GroundingDINO_labels\annotated_barrier_scene_4k_08.png
저장 완료: C:/ai_project01/GroundingDINO_labels\annotated_barrier_scene_4k_food_01.png
저장 완료: C:/ai_project01/GroundingDINO_labels\annotated_barrier_scene_4k_food_02.png
저장 완료: C:/ai_project01/GroundingDINO_labels\annotated_barrier_scene_4k_food_03.png
저장 완료: C:/ai_project01/GroundingDINO_labels\annotated_barrier_scene_4k_food_04.png
저장 완료: C:/ai_project01/GroundingDINO_labels\

In [68]:
annotated = cv2.cvtColor(image_source.copy(),cv2.COLOR_RGB2BGR)
image_height,image_width = image_source.shape[:2]
print(f"image_height={image_height}, image_width={image_width}")

image_height=768, image_width=1024


In [69]:
annotated=cv2.cvtColor(image_source.copy(),cv2.COLOR_RGB2BGR)

In [70]:
for box, label in zip(boxes, phrases):
    # 중심 좌표와 크기를 이미지 크기에 맞게 실제 픽셀 값으로 변환합니다.
    cx, cy, w, h = box
    cx *= image_width
    cy *= image_height
    w *= image_width
    h *= image_height

    # 중심 좌표를 기준으로 바운딩 박스의 왼쪽 위(x_min, y_min)와 오른쪽 아래(x_max, y_max) 좌표를 계산합니다.
    x_min = int(cx - w / 2)
    y_min = int(cy - h / 2)
    x_max = int(cx + w / 2)
    y_max = int(cy + h / 2)

    # 박스 좌표도 출력해서 제대로 그려졌는지 확인합니다.
    print(f"x_min={x_min}, y_min={y_min}")
    print(f"x_max={x_max}, y_max={y_max}")

    # 사각형(바운딩 박스)을 annotated 이미지 위에 그립니다.
    # 색상은 초록색(0, 255, 0), 두께는 2입니다.
    cv2.rectangle(annotated, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

    # 박스 위에 라벨(예: person, car 등)을 텍스트로 표시합니다.
    # 폰트는 기본 OpenCV 폰트이며, 위치는 박스 위, 색상은 초


x_min=430, y_min=623
x_max=480, y_max=693
x_min=100, y_min=547
x_max=205, y_max=630
x_min=534, y_min=633
x_max=602, y_max=713
x_min=142, y_min=487
x_max=168, y_max=509
x_min=269, y_min=371
x_max=288, y_max=386
x_min=587, y_min=548
x_max=690, y_max=594
x_min=85, y_min=494
x_max=126, y_max=530
x_min=138, y_min=446
x_max=164, y_max=463
x_min=191, y_min=486
x_max=227, y_max=528
x_min=178, y_min=445
x_max=197, y_max=461
x_min=832, y_min=405
x_max=850, y_max=418
x_min=223, y_min=408
x_max=251, y_max=427
x_min=0, y_min=710
x_max=67, y_max=767
x_min=854, y_min=560
x_max=921, y_max=607
x_min=132, y_min=700
x_max=231, y_max=767
x_min=946, y_min=499
x_max=971, y_max=517
x_min=249, y_min=387
x_max=273, y_max=403
x_min=460, y_min=426
x_max=478, y_max=443
x_min=875, y_min=456
x_max=896, y_max=473
x_min=270, y_min=426
x_max=294, y_max=448
x_min=851, y_min=420
x_max=870, y_max=434
x_min=921, y_min=611
x_max=995, y_max=666
x_min=0, y_min=644
x_max=31, y_max=691
x_min=403, y_min=423
x_max=415, y_max=440

In [71]:
output_folder = "C:/ai_project01/GroundingDINO_labels"
os.makedirs(output_folder, exist_ok=True)

In [72]:
output_image_path = os.path.join(output_folder, "annotated_image.jpg")
cv2.imwrite(output_image_path, annotated)
print(f"시각화 이미지 저장 완료: {output_image_path}")

시각화 이미지 저장 완료: C:/ai_project01/GroundingDINO_labels\annotated_image.jpg
